# S1 Equities Factor Panel

Builds a **long-format factor panel** for the S1 equities sleeve: daily OHLCV plus engineered alpha features on a point-in-time tradable universe.

**Outputs (two parquets):**

| File | Contents | Used for |
|------|----------|----------|
| `s1_factor_panel_train.parquet` | Chronological **train / research IS** — first `RESEARCH_IS_FRACTION` of unique trading dates | Factor tests (IC, Alphalens) |
| `s1_factor_panel_full.parquet` | **Full sample (census)** — every date in the fetched panel | Later model OOS, backtests, full-history diagnostics |

**Integrity constraints:** features at date `t` use only information available at or before `t`; universe membership is point-in-time (no survivorship bias); forward returns live in separate label columns for training/evaluation only.

**Research split:** `RESEARCH_IS_FRACTION` is the single configurable split constant (default `0.70`). The notebook splits on sorted unique trading dates (never by row count). Factor notebooks load the **train** parquet only and must not re-split it.

Factor hypotheses under test are logged in `02_research/hypothesis_log.md`.

### What is a panel?

In this notebook, **panel** means a pandas **DataFrame** — the same tabular object you would normally call a dataframe. We use *panel* when the table has a repeated cross-section over time: one row per `(date, ticker)` with columns for prices, factors, and labels.

That layout is also called **long format** (stacked dates and tickers) as opposed to **wide format** (dates as rows, tickers as columns), which some libraries use only at export time. Throughout this notebook, `panel` is just the variable name for the working DataFrame.

## 0. Imports & Config

Project paths, cache locations, date range, universe size, dual parquet output paths (`*_train` / `*_full`), and the single chronological research split constant `RESEARCH_IS_FRACTION`.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

from data.ingestion.equity_fetcher import DEFAULT_CACHE_DIR, fetch_ohlcv, fetch_top_n_equities
from data.processing.feature_implementation.beta import add_rolling_beta, market_return_frame

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
# ROOT is then the trading_portfolio directory (used for project-wide paths like ALT_DATA_CACHE_DIR).
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
    
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
TRAIN_OUTPUT_PATH = os.path.join(NOTEBOOK_DIR, "s1_factor_panel_train.parquet")
FULL_OUTPUT_PATH = os.path.join(NOTEBOOK_DIR, "s1_factor_panel_full.parquet")
ALT_DATA_CACHE_DIR = os.path.join(ROOT, "01_data", "cache", "alternative_data")

START_DATE = "2020-01-02"
END_DATE = None
UNIVERSE_N = 100
LOOKBACK_DAYS = 20
BETA_WINDOWS = [20]  # ints only; add e.g. 60 later — each window gets its own column set

# Single source of truth for the chronological factor-research train / holdout split.
RESEARCH_IS_FRACTION = 0.70

### Survivorship bias and yfinance download failures

`fetch_top_n_equities` ranks a **point-in-time** S&P 500 universe, then downloads OHLCV via yfinance. You may see warnings such as `possibly delisted; no timezone found` or `no price data found` for names that were in the universe on the panel start date but are no longer available from Yahoo.

**What this means**

- **Delisted / acquired tickers** (e.g. ATVI, XLNX): Yahoo often removes history. Those names can rank in the top-N by dollar volume but are silently dropped from the downloaded panel — a **survivor bias** toward names Yahoo still serves.
- **Symbol-format mismatches** (e.g. `BRK.B`, `BF.B`): fixed in `equity_fetcher` by mapping canonical `.` tickers to Yahoo `-` symbols and mapping back on output.
- **Transient Yahoo errors** (e.g. BK): can poison the ranking cache if the empty result is cached; delete affected cache files under `01_data/cache/` and re-run.

Ticker mapping fixes symbol-format issues but **does not** restore delisted history. Treat the panel as potentially survivor-biased until a PIT-capable vendor replaces yfinance for dead names.


## 1. Data Loading

Fetch or load daily OHLCV for the point-in-time S&P 500 universe (top-N by trailing dollar volume as of the panel start).

In [2]:
panel = fetch_top_n_equities(
    n=UNIVERSE_N,
    start_date=START_DATE,
    end_date=END_DATE,
    lookback_days=LOOKBACK_DAYS,
    cache_dir=DEFAULT_CACHE_DIR,
)

print(f"shape: {panel.shape}")
print(f"tickers: {panel['ticker'].nunique()}")
print(f"date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
panel.head()

$ADS: possibly delisted; no timezone found
$ABMD: possibly delisted; no timezone found
$ABC: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$ANSS: possibly delisted; no timezone found
$ARNC: possibly delisted; no timezone found
$ANTM: possibly delisted; no timezone found
$ALXN: possibly delisted; no timezone found

8 Failed downloads:
['ADS', 'ABMD', 'ABC', 'AGN', 'ANSS', 'ARNC', 'ANTM', 'ALXN']: possibly delisted; no timezone found
$BLL: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2019-12-05 -> 2020-01-03) (Yahoo error = "Data doesn't exist for startDate = 1575522000, endDate = 1578027600")
$CMA: possibly delisted; no timezone found
$CERN: possibly delisted; no timezone found
$ATVI: possibly delisted; no timezone found

5 Failed downloads:
['BLL', 'CMA', 'CERN', 'ATVI']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2019-12-05 -> 2020-01-03) (Yahoo error = "Data doesn'

shape: (164300, 7)
tickers: 100
date range: 2020-01-02 → 2026-07-17


,date,ticker,open,high,low,close,volume
0,2020-01-02,AAPL,71.344062,72.394093,71.091191,72.333885,135480400
1,2020-01-02,ABBV,67.852721,68.225955,67.418550,68.210724,5639200
2,2020-01-02,ABT,75.995215,76.789960,75.765627,76.781128,4969000
3,2020-01-02,ACN,189.265376,190.216817,187.425318,188.628082,2431100
4,2020-01-02,ADBE,330.000000,334.480011,329.170013,334.429993,1990100


## 2. Integration of Alternative Data

Merge cached alternative-data features (e.g. news sentiment) onto the OHLCV panel. Align with `merge_asof(..., direction='backward')` so each row at date `t` only sees data published on or before `t`.

### 2.1 Beta

Rolling OLS of each stock's daily log returns on SPY, estimated **per ticker** over every window in `BETA_WINDOWS`. Features at date `t` use returns through the close of `t` only (`min_periods=W`; partial windows → NaN). No `bfill` on returns or regression outputs.

**Model:** `r_{i,t} = alpha + beta * r_{SPY,t} + epsilon_t`

**Benchmark:** SPY via `fetch_ohlcv("SPY", START_DATE, END_DATE, cache_dir=DEFAULT_CACHE_DIR)`.

**Returns:** `ln(close_t / close_{t-1})`; the first bar of each series is NaN.

**Attached columns** (alpha / beta / r2 only — idio vol is built later in H-003):

| Column naming | When |
|---------------|------|
| `alpha`, `beta`, `r2` | `len(BETA_WINDOWS) == 1` |
| `alpha_{W}`, `beta_{W}`, `r2_{W}` | `len(BETA_WINDOWS) > 1` |

**Not stored here:** `idio_vol` (residual std) and per-day residuals `epsilon_t`. Idiosyncratic vol is implemented in `data.processing.feature_implementation.idiosyncratic_vol` and tested in `02_research/notebooks/factor_tests/H-003_idiosyncratic_vol.ipynb`.

Uses `add_rolling_beta` from `data.processing.feature_implementation.beta` for reproducibility in walk-forward and live pipelines. Supports H-004 (beta); H-003 factor engineering is deferred.


### 2.2 Size & Value 

### 2.3 Sentiment

## 3. Panel Export (train + full census)

Write **two** parquets from the completed daily panel:

1. **Full sample (census)** — every `(date, ticker)` row → `s1_factor_panel_full.parquet`.
2. **Train / research IS** — first `RESEARCH_IS_FRACTION` of **sorted unique trading dates** → `s1_factor_panel_train.parquet`.

The split is chronological by unique dates (never by row count). All tickers on an included date stay together. Factor-test notebooks consume the **train** file only.

In [3]:
if not 0.0 < RESEARCH_IS_FRACTION < 1.0:
    raise ValueError("RESEARCH_IS_FRACTION must be strictly between 0 and 1")

# Split on unique trading dates (never by row count)
unique_dates = pd.Index(pd.to_datetime(panel["date"]).dropna().unique()).sort_values()
if len(unique_dates) < 2:
    raise ValueError("panel must contain at least two unique dates for a train/holdout split")

n_is_dates = int(len(unique_dates) * RESEARCH_IS_FRACTION)
if n_is_dates < 1 or n_is_dates >= len(unique_dates):
    raise ValueError("RESEARCH_IS_FRACTION leaves an empty train or holdout date range")

is_dates = unique_dates[:n_is_dates]
holdout_dates = unique_dates[n_is_dates:]
train_panel = panel[pd.to_datetime(panel["date"]).isin(is_dates)].copy()
full_panel = panel.copy()

full_panel.to_parquet(FULL_OUTPUT_PATH, index=False)
train_panel.to_parquet(TRAIN_OUTPUT_PATH, index=False)

print(f"saved full census: {FULL_OUTPUT_PATH}")
print(f"  shape: {full_panel.shape}")
print(f"  dates: {unique_dates[0].date()} → {unique_dates[-1].date()} ({len(unique_dates)})")
print(f"saved train / research IS: {TRAIN_OUTPUT_PATH}")
print(f"  split fraction: {RESEARCH_IS_FRACTION:.2f}")
print(f"  shape: {train_panel.shape}")
print(f"  IS dates: {is_dates[0].date()} → {is_dates[-1].date()} ({len(is_dates)})")
print(
    f"  excluded holdout dates: {holdout_dates[0].date()} → "
    f"{holdout_dates[-1].date()} ({len(holdout_dates)})"
)
print(f"columns: {train_panel.columns.tolist()}")
train_panel.head()

saved full census: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_factor_panel_full.parquet
  shape: (164300, 7)
  dates: 2020-01-02 → 2026-07-17 (1643)
saved train / research IS: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_factor_panel_train.parquet
  split fraction: 0.70
  shape: (115000, 7)
  IS dates: 2020-01-02 → 2024-07-29 (1150)
  excluded holdout dates: 2024-07-30 → 2026-07-17 (493)
columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']


,date,ticker,open,high,low,close,volume
0,2020-01-02,AAPL,71.344062,72.394093,71.091191,72.333885,135480400
1,2020-01-02,ABBV,67.852721,68.225955,67.418550,68.210724,5639200
2,2020-01-02,ABT,75.995215,76.789960,75.765627,76.781128,4969000
3,2020-01-02,ACN,189.265376,190.216817,187.425318,188.628082,2431100
4,2020-01-02,ADBE,330.000000,334.480011,329.170013,334.429993,1990100
